# Document Chunking Masterclass — From Fixed-Size to Agentic

**Goal:** understand every major chunking strategy used in production RAG systems,
by running each one on the *same* real documents and comparing what comes out.

**Inputs used throughout this notebook:**
- `data/clinical_trial_protocol.pdf` — a 5-page clinical trial protocol (headings, lists, numbered sections)
- `data/data_handling_policy.docx` — a compliance policy document (headings, bullet lists)

**Roadmap:**
1. Setup — load and inspect the raw text of both documents
2. Fixed-size chunking
3. Sentence / Paragraph chunking
4. Recursive character chunking
5. Structure-aware chunking
6. Semantic chunking
7. LLM / Agentic chunking
8. Build a vector store per strategy + run 5 fixed questions against each
9. Final comparison table + healthcare-specific recommendations

We build this one section at a time — run each cell, look at the actual output,
*then* move to the next strategy. Don't skip ahead; the whole point is to build
intuition for *why* chunk boundaries end up where they do.


## Section 0 — Setup: Loading Our Two Real Documents

Before we can chunk anything, we need plain text. Real documents are messy:
- PDFs store text as positioned glyphs, not paragraphs — extraction can merge or split lines oddly.
- DOCX files store text in "runs" inside XML — headings and bullets carry structure metadata we can use later (Section 5).

**Production note:** in a real pipeline this "extraction" step is its own component,
often the least glamorous and most bug-prone part of RAG. Garbage extraction → garbage chunks →
garbage retrieval, no matter how good your chunking strategy is downstream.


In [ ]:
# If running this notebook fresh, install dependencies first (uncomment):
# %pip install pypdf python-docx langchain langchain-text-splitters tiktoken \
#              sentence-transformers faiss-cpu scikit-learn nltk

import os
import textwrap

DATA_DIR = "../data"
PDF_PATH = os.path.join(DATA_DIR, "clinical_trial_protocol.pdf")
DOCX_PATH = os.path.join(DATA_DIR, "data_handling_policy.docx")

print("PDF exists:", os.path.exists(PDF_PATH))
print("DOCX exists:", os.path.exists(DOCX_PATH))


### 0.1 Extract text from the PDF

In [2]:
from pypdf import PdfReader


def load_pdf_text(path: str) -> str:
    """Extract raw text from a PDF, page by page, keeping a page marker.
    We keep '[[PAGE:n]]' markers so structure-aware chunking (Section 5)
    can later use page boundaries if it needs to.
    """
    reader = PdfReader(path)
    pages_text = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        pages_text.append(f"[[PAGE:{i+1}]]\n{text}")
    return "\n".join(pages_text)

pdf_text = load_pdf_text(PDF_PATH)

print(f"Total characters extracted: {len(pdf_text):,}")
print(f"Approx. words: {len(pdf_text.split()):,}")
print("\n--- First 600 characters ---\n")
print(pdf_text[:600])


Total characters extracted: 10,653
Approx. words: 1,536

--- First 600 characters ---

[[PAGE:1]]
Clinical Trial Protocol
Protocol Number: CTP-2026-0142
Study Title: A Phase II, Randomized, Double-Blind, Placebo-Controlled Study of Oral Compound AX-119
in Adult Patients with Moderate-to-Severe Rheumatoid Arthritis
Sponsor: Meridian Biotherapeutics Inc.
Version: 3.1 | Date: 15-Jan-2026
1. Study Objectives
1.1 Primary Objective
The primary objective of this study is to evaluate the efficacy of AX-119 compared to placebo in reducing
disease activity, as measured by the American College of Rheumatology 20% response criteria (ACR20),
at Week 24 in adult patients with moderate-to-seve


### 0.2 Extract text from the DOCX

In [3]:
from docx import Document as DocxDocument


def load_docx_text(path: str):
    """Extract text paragraph by paragraph, keeping each paragraph's
    style name (e.g. 'Heading 1', 'List Bullet', 'Normal'). We return a
    list of (style, text) tuples — this structure is what Section 5
    (structure-aware chunking) will use directly, and we also build a
    plain joined string for the strategies that just want raw text.
    """
    doc = DocxDocument(path)
    paragraphs = []
    for para in doc.paragraphs:
        text = para.text.strip()
        if not text:
            continue
        # Real-world gotcha: para.style can be None if the style reference
        # doesn't resolve cleanly. Don't let that crash extraction -- fall
        # back to "Normal" and move on. Bad extraction should degrade
        # gracefully, not blow up the whole pipeline.
        style_name = para.style.name if para.style is not None else "Normal"
        paragraphs.append((style_name, text))
    return paragraphs

docx_paragraphs = load_docx_text(DOCX_PATH)
docx_text = "\n".join(text for _, text in docx_paragraphs)

print(f"Total paragraphs: {len(docx_paragraphs)}")
print(f"Total characters: {len(docx_text):,}")
print("\n--- First 10 (style, text) pairs ---\n")
for style, text in docx_paragraphs[:10]:
    print(f"[{style}] {textwrap.shorten(text, width=80)}")


Total paragraphs: 31
Total characters: 4,707

--- First 10 (style, text) pairs ---

[Title] Clinical Data Handling and Privacy Policy
[Normal] Document ID: POL-COMP-014 | Version 2.3 | Effective Date: 01-Feb-2026
[Heading 1] 1. Purpose
[Normal] This policy establishes the requirements for the collection, storage, [...]
[Heading 1] 2. Scope
[Normal] This policy covers protected health information (PHI), personally [...]
[Heading 1] 3. Regulatory Framework
[Normal] Data handling practices under this policy must comply with the following [...]
[List Paragraph] The Health Insurance Portability and Accountability Act (HIPAA) for data [...]
[List Paragraph] The General Data Protection Regulation (GDPR) for data collected within [...]


### 0.3 A quick sanity check

Before chunking, always eyeball the raw extracted text. Two things to check:

1. **Did extraction lose anything obvious?** (tables collapsing, headers merging into body text, garbled characters)
2. **Roughly how big is the document?** — this tells you whether "chunking" even matters (a 2-page doc barely needs it) and what chunk size might be sensible relative to total length.


In [4]:
print("=" * 60)
print("PDF  — clinical_trial_protocol.pdf")
print("=" * 60)
print(f"Characters: {len(pdf_text):,} | Words: {len(pdf_text.split()):,} | Pages: {pdf_text.count('[[PAGE:')}")

print()
print("=" * 60)
print("DOCX — data_handling_policy.docx")
print("=" * 60)
print(f"Characters: {len(docx_text):,} | Words: {len(docx_text.split()):,} | Paragraphs: {len(docx_paragraphs)}")


PDF  — clinical_trial_protocol.pdf
Characters: 10,653 | Words: 1,536 | Pages: 5

DOCX — data_handling_policy.docx
Characters: 4,707 | Words: 681 | Paragraphs: 31


**What we have now:**
- `pdf_text` — full plain text of the clinical trial protocol, with `[[PAGE:n]]` markers
- `docx_text` — full plain text of the compliance policy, paragraphs joined by newline
- `docx_paragraphs` — list of `(style_name, text)` — keeps Word's own structure info intact

Everything from Section 1 onward will chunk these same variables, so every
strategy is compared on identical input — that's what makes the final
comparison meaningful.

---
✅ **Checkpoint — Section 0 complete.**
Confirm this looks right, and we'll move to **Section 1: Fixed-size chunking**.
